In [1]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_BGR
import cv2
import numpy as np
import time


class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, i):
        if i not in self.parent:
            self.parent[i] = i
            return i
        if self.parent[i] == i:
            return i
        self.parent[i] = self.find(self.parent[i]) # Path compression
        return self.parent[i]

    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j

# --- 1. Load Base Overlay ---
base = BaseOverlay("base.bit")
hdmi_out = base.video.hdmi_out

# --- 2. Configure HDMI ---
mode = VideoMode(1280, 720, 24)
hdmi_out.configure(mode, pixelformat=PIXEL_BGR)
hdmi_out.start()

# --- 3. Load Image ---
input_filename = "crosses.jpg"
input_path = f"images/{input_filename}"

try:
    img = cv2.imread(input_path)
    if img is None:
        raise FileNotFoundError(f"Could not find image at {input_path}")

    print(f"Image loaded. Starting Processing...")

    # PRE-PROCESSING: Resize DOWN for Pure Python speed
    # We process at 320x180, then resize up for the 1280x720 display
    process_w, process_h = 320, 180 
    small_img = cv2.resize(img, (process_w, process_h))
    
    # Convert to Grayscale
    gray = cv2.cvtColor(small_img, cv2.COLOR_BGR2GRAY)
    
    # --- CRITICAL FIX: INVERSE THRESHOLD ---
    # We use THRESH_BINARY_INV. 
    # This turns Black pixels (objects) into 1s, and White pixels (background) into 0s.
    # We also use THRESH_OTSU to automatically find the best split value.
    thresh_val, binary = cv2.threshold(gray, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    print(f"Thresholding complete. Used value: {thresh_val}")

    # --- PURE PYTHON TWO-PASS ALGORITHM ---
    
    labels = np.zeros((process_h, process_w), dtype=int)
    next_label = 1
    uf = UnionFind()
    
    # === PASS 1: Assign Labels & Record Equivalences ===
    print("Starting Pass 1...")
    t0 = time.time()

    for r in range(process_h):
        for c in range(process_w):
            if binary[r, c] == 1:
                # Neighbors: West (left) and North (up)
                west  = labels[r, c-1] if c > 0 else 0
                north = labels[r-1, c] if r > 0 else 0

                if west == 0 and north == 0:
                    # New component found
                    labels[r, c] = next_label
                    uf.find(next_label) 
                    next_label += 1
                elif west != 0 and north == 0:
                    labels[r, c] = west
                elif west == 0 and north != 0:
                    labels[r, c] = north
                else:
                    # Connected to both: assign one and record equivalence
                    labels[r, c] = min(west, north)
                    if west != north:
                        uf.union(west, north)

    t1 = time.time()
    pass1_time = t1 - t0
    print(f"Pass 1: {pass1_time:.4f} sec")

    # === PASS 2: Flatten/Resolve Labels ===
    print("Starting Pass 2...")
    t2 = time.time()

    for r in range(process_h):
        for c in range(process_w):
            if labels[r, c] > 0:
                labels[r, c] = uf.find(labels[r, c])

    t3 = time.time()
    pass2_time = t3 - t2
    print(f"Pass 2: {pass2_time:.4f} sec")

    # --- COLORIZATION ---
    print("Colorizing components...")
    
    # Create random colors. Index 0 is forced to Black (Background)
    colors = np.random.randint(0, 255, size=(next_label + 1, 3), dtype=np.uint8)
    colors[0] = [0, 0, 0] 

    # Apply colors using numpy indexing
    output_small = colors[labels]

    # --- RESIZE TO HDMI RESOLUTION ---
    # Scale back up to 1280x720 using Nearest Neighbor to keep edges sharp
    final_output = cv2.resize(output_small, (1280, 720), interpolation=cv2.INTER_NEAREST)

    # --- SAVE AND DISPLAY ---
    save_name = f"images/pure_python/pure_python_{input_filename.split('.')[0]}_labeled.jpg"
    cv2.imwrite(save_name, final_output)

    outframe = hdmi_out.newframe()
    outframe[:] = final_output
    hdmi_out.writeframe(outframe)
    
    print(f"Success! Output displayed and saved to {save_name}")

except Exception as e:
    print(f"Error: {e}")

Image loaded. Starting Processing...
Thresholding complete. Used value: 131.0
Starting Pass 1...
Pass 1: 2.5204 sec
Starting Pass 2...
Pass 2: 0.5052 sec
Colorizing components...
Success! Output displayed and saved to images/pure_python/pure_python_crosses_labeled.jpg
